In [1]:
import pandas as pd

clauses = [
    # Bond period clauses
    ("Employee must serve a bond period of 2 years or pay Rs 1,00,000", "bond"),
    ("A training bond of 18 months is mandatory for all new joiners", "bond"),
    ("Failure to complete bond period will result in recovery of training costs", "bond"),
    ("Employee agrees to serve minimum 2 years or refund training expenses", "bond"),
    ("Bond period of 24 months applicable from date of joining", "bond"),
    ("Breaking bond before completion will incur penalty of 2 months salary", "bond"),
    ("Employee must complete bond period before resignation", "bond"),
    ("Training bond of Rs 50,000 applicable if leaving before 1 year", "bond"),

    # Notice period clauses
    ("Employee must serve 90 days notice period before resignation", "notice"),
    ("A notice period of 3 months is required from either party", "notice"),
    ("Failure to serve notice period will result in salary deduction", "notice"),
    ("Notice period can be bought out at 3 months salary", "notice"),
    ("Either party may terminate with 60 days written notice", "notice"),
    ("30 days notice required for all employees below manager level", "notice"),
    ("Notice period of 2 months mandatory for senior positions", "notice"),
    ("Company may waive notice period at its discretion", "notice"),

    # Non-compete clauses
    ("Employee shall not join any competitor for 1 year after leaving", "non_compete"),
    ("Non-compete clause applicable for 2 years post employment", "non_compete"),
    ("Employee cannot work in same industry for 12 months after resignation", "non_compete"),
    ("Joining a competing firm within 6 months will result in legal action", "non_compete"),
    ("Employee agrees not to solicit clients for 1 year after leaving", "non_compete"),
    ("Non-compete applicable within same geographic region for 2 years", "non_compete"),
    ("Employee shall not engage with competitors during and after employment", "non_compete"),
    ("Violation of non-compete will result in damages and legal proceedings", "non_compete"),

    # IP ownership clauses
    ("All work produced during employment is property of the company", "ip_ownership"),
    ("Any invention or creation during employment belongs to employer", "ip_ownership"),
    ("Employee assigns all intellectual property rights to the company", "ip_ownership"),
    ("Code, designs and innovations created during tenure are company property", "ip_ownership"),
    ("Employee waives all rights to work produced during employment", "ip_ownership"),
    ("Company owns all patents and copyrights from employee work", "ip_ownership"),
    ("All work product including side projects belongs to employer", "ip_ownership"),
    ("Employee must disclose all inventions made during employment", "ip_ownership"),

    # Normal/standard clauses
    ("Employee will receive 12 days of paid annual leave per year", "normal"),
    ("Salary will be credited on the last working day of each month", "normal"),
    ("Employee is entitled to medical insurance coverage from day one", "normal"),
    ("Performance review will be conducted annually every April", "normal"),
    ("Employee will receive provident fund contributions as per law", "normal"),
    ("Work hours are 9am to 6pm Monday to Friday", "normal"),
    ("Employee is entitled to 10 days sick leave per year", "normal"),
    ("Salary increment will be based on annual performance review", "normal"),
]

df_clauses = pd.DataFrame(clauses, columns=['clause', 'label'])
print("Dataset shape:", df_clauses.shape)
print("\nLabel distribution:")
print(df_clauses['label'].value_counts())

Dataset shape: (40, 2)

Label distribution:
label
bond            8
notice          8
non_compete     8
ip_ownership    8
normal          8
Name: count, dtype: int64


In [2]:
more_clauses = [
    # More bond variations
    ("Employee is required to complete a service bond of 3 years", "bond"),
    ("Leaving before bond completion requires payment of Rs 2,00,000", "bond"),
    ("Bond amount will be recovered from full and final settlement", "bond"),
    ("Service agreement of 2 years mandatory after training period", "bond"),

    # More notice variations  
    ("Immediate resignation without notice will lead to salary forfeiture", "notice"),
    ("Notice period buyout option available at current CTC divided by 12", "notice"),
    ("Employee serving notice period must complete all handover tasks", "notice"),
    ("Company reserves right to accept or reject notice period buyout", "notice"),

    # More non-compete variations
    ("Employee must not directly or indirectly work for competitors", "non_compete"),
    ("Soliciting former colleagues after leaving is strictly prohibited", "non_compete"),
    ("Employee cannot start competing business for 2 years post exit", "non_compete"),
    ("Geographic restriction applies to entire country for non-compete", "non_compete"),

    # More IP variations
    ("Side projects related to company business require prior approval", "ip_ownership"),
    ("Employee must sign IP assignment agreement on joining date", "ip_ownership"),
    ("All client data and business intelligence remains company property", "ip_ownership"),
    ("Employee has no claim over tools or software built during tenure", "ip_ownership"),

    # More normal variations
    ("Employee will be eligible for gratuity after 5 years of service", "normal"),
    ("Company will reimburse travel expenses as per policy", "normal"),
    ("Employee is entitled to maternity or paternity leave as per law", "normal"),
    ("Dress code is business casual from Monday to Thursday", "normal"),
    ("Employee will receive laptop and necessary equipment on day one", "normal"),
    ("Annual bonus will be paid based on company and individual performance", "normal"),
]

df_more = pd.DataFrame(more_clauses, columns=['clause', 'label'])
df_clauses = pd.concat([df_clauses, df_more], ignore_index=True)

print("Expanded dataset shape:", df_clauses.shape)
print("\nLabel distribution:")
print(df_clauses['label'].value_counts())

df_clauses.to_csv('offer_clauses.csv', index=False)
print("\nSaved!")

Expanded dataset shape: (62, 2)

Label distribution:
label
normal          14
bond            12
notice          12
non_compete     12
ip_ownership    12
Name: count, dtype: int64

Saved!


In [3]:
import re
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

def clean_clause(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df_clauses['cleaned_clause'] = df_clauses['clause'].apply(clean_clause)
print("Sample cleaned clause:")
print(df_clauses['cleaned_clause'][0])

Sample cleaned clause:
employee must serve bond period years pay rs


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Split data
X = df_clauses['cleaned_clause']
y = df_clauses['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 49
Testing samples: 13


In [5]:
# Vectorize text
vectorizer = TfidfVectorizer(max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train classifier
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train_vec, y_train)

print("Model trained!")

# Evaluate
y_pred = classifier.predict(X_test_vec)
print("\nAccuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

Model trained!

Accuracy: 92.31 %

Detailed Report:
              precision    recall  f1-score   support

        bond       1.00      1.00      1.00         3
ip_ownership       0.67      1.00      0.80         2
 non_compete       1.00      1.00      1.00         3
      normal       1.00      0.67      0.80         3
      notice       1.00      1.00      1.00         2

    accuracy                           0.92        13
   macro avg       0.93      0.93      0.92        13
weighted avg       0.95      0.92      0.92        13



In [6]:
# See actual vs predicted for test set
results = pd.DataFrame({
    'clause': X_test.values,
    'actual': y_test.values,
    'predicted': y_pred
})

# Show only wrong predictions
wrong = results[results['actual'] != results['predicted']]
print("Wrong predictions:")
print(wrong[['clause', 'actual', 'predicted']].to_string())

Wrong predictions:
                                       clause  actual     predicted
4  dress code business casual monday thursday  normal  ip_ownership


In [7]:
import pickle

pickle.dump(classifier, open('offer_classifier.pkl', 'wb'))
pickle.dump(vectorizer, open('offer_vectorizer.pkl', 'wb'))

print("Classifier saved!")
print("Vectorizer saved!")

Classifier saved!
Vectorizer saved!


In [8]:
def predict_clause(text):
    cleaned = clean_clause(text)
    vectorized = vectorizer.transform([cleaned])
    prediction = classifier.predict(vectorized)[0]
    confidence = classifier.predict_proba(vectorized).max() * 100
    return prediction, round(confidence, 2)

# Test with new clauses never seen before
test_clauses = [
    "Employee must serve 2 year bond or pay Rs 1,50,000",
    "You cannot join any competitor for 1 year after leaving",
    "All code written during employment belongs to the company",
    "Employee will receive 15 days of casual leave annually",
    "90 days notice period required before resignation"
]

print("Predictions on new clauses:\n")
for clause in test_clauses:
    label, confidence = predict_clause(clause)
    print(f"Clause: {clause}")
    print(f"Predicted: {label} (confidence: {confidence}%)\n")

Predictions on new clauses:

Clause: Employee must serve 2 year bond or pay Rs 1,50,000
Predicted: bond (confidence: 36.84%)

Clause: You cannot join any competitor for 1 year after leaving
Predicted: non_compete (confidence: 40.17%)

Clause: All code written during employment belongs to the company
Predicted: ip_ownership (confidence: 39.62%)

Clause: Employee will receive 15 days of casual leave annually
Predicted: normal (confidence: 44.79%)

Clause: 90 days notice period required before resignation
Predicted: notice (confidence: 53.47%)



In [9]:
offer_engine_code = '''
import re
import pickle
import numpy as np
from nltk.corpus import stopwords
import nltk
nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))

classifier = pickle.load(open("offer_classifier.pkl", "rb"))
vectorizer = pickle.load(open("offer_vectorizer.pkl", "rb"))

RISK_LABELS = {
    "bond": {
        "risk": "High Risk",
        "color": "red",
        "explanation": "This clause binds you to the company for a fixed period. Leaving early may cost you money."
    },
    "non_compete": {
        "risk": "High Risk", 
        "color": "red",
        "explanation": "This restricts where you can work after leaving. May limit your career options."
    },
    "ip_ownership": {
        "risk": "Medium Risk",
        "color": "amber",
        "explanation": "The company claims ownership of work you produce. Check if this includes personal side projects."
    },
    "notice": {
        "risk": "Medium Risk",
        "color": "amber", 
        "explanation": "Long notice periods can make it hard to switch jobs quickly."
    },
    "normal": {
        "risk": "Standard",
        "color": "green",
        "explanation": "This appears to be a standard employment clause."
    }
}

def clean_clause(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z\\s]", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

def analyze_clause(text):
    cleaned = clean_clause(text)
    vectorized = vectorizer.transform([cleaned])
    label = classifier.predict(vectorized)[0]
    confidence = classifier.predict_proba(vectorized).max() * 100
    
    risk_info = RISK_LABELS[label]
    
    return {
        "clause": text,
        "label": label,
        "risk": risk_info["risk"] if confidence >= 40 else "Uncertain",
        "color": risk_info["color"] if confidence >= 40 else "gray",
        "explanation": risk_info["explanation"] if confidence >= 40 else "Could not determine risk level confidently. Review manually.",
        "confidence": round(confidence, 2)
    }

def analyze_offer_letter(offer_text):
    sentences = [s.strip() for s in offer_text.split(".") if len(s.strip()) > 20]
    results = [analyze_clause(s) for s in sentences]
    
    high_risk = [r for r in results if r["risk"] == "High Risk"]
    medium_risk = [r for r in results if r["risk"] == "Medium Risk"]
    
    return {
        "total_clauses": len(results),
        "high_risk_count": len(high_risk),
        "medium_risk_count": len(medium_risk),
        "clauses": results
    }
'''

with open('offer_engine.py', 'w') as f:
    f.write(offer_engine_code)

print("offer_engine.py saved!")

offer_engine.py saved!


In [10]:
sample_offer = """
Employee will receive a salary of Rs 8,00,000 per annum.
Employee must serve a bond period of 2 years or pay Rs 1,00,000.
Work hours are 9am to 6pm Monday to Friday.
Employee shall not join any competitor for 1 year after leaving.
All code and work produced during employment belongs to the company.
Employee must serve 90 days notice period before resignation.
Employee is entitled to 12 days paid leave annually.
"""

import importlib.util
spec = importlib.util.spec_from_file_location("offer_engine", "offer_engine.py")
offer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(offer)

result = offer.analyze_offer_letter(sample_offer)

print(f"Total clauses analyzed: {result['total_clauses']}")
print(f"High risk clauses: {result['high_risk_count']}")
print(f"Medium risk clauses: {result['medium_risk_count']}")
print("\nDetailed breakdown:")
for clause in result['clauses']:
    print(f"\n[{clause['risk']}] {clause['clause'][:60]}...")
    print(f"   Label: {clause['label']} | Confidence: {clause['confidence']}%")
    print(f"   {clause['explanation']}")

Total clauses analyzed: 7
High risk clauses: 2
Medium risk clauses: 2

Detailed breakdown:

[Standard] Employee will receive a salary of Rs 8,00,000 per annum...
   Label: normal | Confidence: 44.14%
   This appears to be a standard employment clause.

[High Risk] Employee must serve a bond period of 2 years or pay Rs 1,00,...
   Label: bond | Confidence: 46.61%
   This clause binds you to the company for a fixed period. Leaving early may cost you money.

[Uncertain] Work hours are 9am to 6pm Monday to Friday...
   Label: normal | Confidence: 38.33%
   Could not determine risk level confidently. Review manually.

[High Risk] Employee shall not join any competitor for 1 year after leav...
   Label: non_compete | Confidence: 41.64%
   This restricts where you can work after leaving. May limit your career options.

[Medium Risk] All code and work produced during employment belongs to the ...
   Label: ip_ownership | Confidence: 51.55%
   The company claims ownership of work you produce. C